In [ ]:
# --- imports ---
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# SETTINGS - edit these values as needed
# Keep the anchor symbol last in the list.
# ============================================================
sorted_symbols_list = ["CWB", "ICVT"]
moving_avg_days = 20
commission_per_share = 0.005
dollar_constant = 100_000
trading_days_per_year = 252
input_directory = Path("with_positions")
output_directory = Path("with_stats")


def add_statistics(df):
    anchor = sorted_symbols_list[-1]
    non_anchor = sorted_symbols_list[0]

    # Size each leg to approximately the configured dollar constant.
    anchor_average = f"yday {anchor} / {anchor} moving avg"
    anchor_shares = f"{anchor} shares per unit"
    df[anchor_shares] = (
        df[anchor_average] * dollar_constant / df[f"to {anchor} price"]
    ).round()

    for symbol in sorted_symbols_list:
        shares_per_unit = f"{symbol} shares per unit"
        if symbol != anchor:
            prior_average = f"yday {anchor} / {symbol} moving avg"
            df[shares_per_unit] = (df[anchor_shares] * df[prior_average]).round()

    # A position of 1 is long the non-anchor spread; -1 is short it.
    for symbol in sorted_symbols_list:
        leg_sign = 1 if symbol == non_anchor else -1
        shares_per_unit = f"{symbol} shares per unit"
        target_shares = f"{symbol} target shares"
        current_shares = f"{symbol} current shares"

        df[target_shares] = df["new_position"] * leg_sign * df[shares_per_unit]
        df[current_shares] = df[target_shares].shift(1)

        df[f"{symbol} investment amount"] = (
            df[current_shares] * df[f"from {symbol} price"]
        )
        df[f"{symbol} shares to trade"] = (
            df[target_shares] - df[current_shares]
        )
        df[f"{symbol} shares to buy"] = (
            df[f"{symbol} shares to trade"].clip(lower=0)
        )
        df[f"{symbol} shares to sell"] = (
            -df[f"{symbol} shares to trade"].clip(upper=0)
        )
        df[f"{symbol} daily commission"] = (
            -df[f"{symbol} shares to trade"].abs() * commission_per_share
        )
        df[f"{symbol} daily profit"] = (
            df[current_shares] * df[f"{symbol} cod"]
        )

    investment_columns = [
        f"{symbol} investment amount" for symbol in sorted_symbols_list
    ]
    commission_columns = [
        f"{symbol} daily commission" for symbol in sorted_symbols_list
    ]
    profit_columns = [
        f"{symbol} daily profit" for symbol in sorted_symbols_list
    ]
    shares_traded_columns = [
        f"{symbol} shares to trade" for symbol in sorted_symbols_list
    ]

    df["gross investment amount"] = df[investment_columns].abs().sum(axis=1)
    df["net investment amount"] = df[investment_columns].sum(axis=1)
    df["daily commission"] = df[commission_columns].sum(axis=1)
    df["daily gross profit"] = df[profit_columns].sum(axis=1)
    df["daily net profit"] = df["daily gross profit"] + df["daily commission"]
    df["cumulative net profit"] = df["daily net profit"].cumsum()
    df["drawdown"] = (
        df["cumulative net profit"]
        - df["cumulative net profit"].cummax()
    )

    valid_rows = df.index >= moving_avg_days
    daily_net_profit = df.loc[valid_rows, "daily net profit"]
    average_daily_investment = df.loc[
        valid_rows, "gross investment amount"
    ].mean()
    total_net_profit = daily_net_profit.sum()
    number_of_days = daily_net_profit.notna().sum()

    if average_daily_investment > 0 and number_of_days > 0:
        annualized_return = (
            total_net_profit
            / average_daily_investment
            * trading_days_per_year
            / number_of_days
        )
    else:
        annualized_return = np.nan

    daily_std = daily_net_profit.std()
    if daily_std != 0 and not pd.isna(daily_std):
        annualized_sharpe = (
            daily_net_profit.mean() / daily_std * np.sqrt(trading_days_per_year)
        )
    else:
        annualized_sharpe = np.nan

    df["total net profit"] = total_net_profit
    df["average daily investment"] = average_daily_investment
    df["annualized return on avg investment"] = annualized_return
    df["annualized Sharpe"] = annualized_sharpe
    df["maximum drawdown"] = df["drawdown"].min()
    df["position changes"] = df["new_position"].ne(df["current_position"]).sum()
    df["total shares traded"] = df[shares_traded_columns].abs().sum().sum()

    return df


if len(sorted_symbols_list) != 2:
    raise ValueError("This position-statistics notebook requires exactly two symbols")
if not input_directory.exists():
    raise FileNotFoundError(f"Input directory not found: {input_directory}")

input_files = sorted(input_directory.glob("*.csv"))
if not input_files:
    raise FileNotFoundError(f"No CSV files found in {input_directory}")

output_directory.mkdir(parents=True, exist_ok=True)

for input_path in input_files:
    df = pd.read_csv(input_path, index_col=0)
    df.columns = df.columns.str.strip()
    df = add_statistics(df)

    output_path = output_directory / input_path.name
    df.to_csv(output_path)
    print(f"Saved {output_path}")

print(f"finished: processed {len(input_files)} files")
